In [ ]:
 
import re
import numpy as np
import soundfile as sf
from IPython.display import Audio as play,display
from cached_path import cached_path
from hydra.utils import get_class
from omegaconf import OmegaConf
import joblib
 
from f5_tts.infer.utils_infer import (
    cfg_strength,
    cross_fade_duration,
    fix_duration,
    infer_process,
    load_model,
    load_vocoder,
    nfe_step,
    speed,
    sway_sampling_coef,
    target_rms,
)
from f5_tts.infer.utils_xeus import ApplyKmeans, load_xeus_model, extract_units
import warnings
warnings.filterwarnings("ignore")

In [2]:
device = "cuda" # Set to your device
vocoder_name= "vocos"
config_file = "../configs/F5TTS_v1_Base_units_Arabic_6k.yaml"
ckpt_file = "../../../checkpoints/model_150000.pt"
vocab_file = "../../../data/Arabic_training_6k_custom/vocab.txt"
kmeans_path="../../../checkpoints/kmeans_2000.pkl"


 

In [3]:
km_model = joblib.load(kmeans_path)


In [4]:
# load XEUS model

xeus_model = load_xeus_model(device).eval()
apply_kmeans = ApplyKmeans(km_model,device)

In [5]:
# load vocoder

vocoder = load_vocoder(vocoder_name=vocoder_name, device=device)

Download Vocos from huggingface charactr/vocos-mel-24khz


In [6]:
# load TTS model

model_cfg = OmegaConf.load(config_file)
model_cls = get_class(f"f5_tts.model.{model_cfg.model.backbone}")
model_arc = model_cfg.model.arch

ema_model = load_model(
    model_cls,
    model_arc,
    ckpt_file,
    mel_spec_type=vocoder_name,
    vocab_file=vocab_file,
    device=device,
    use_ema=False,
)


vocab :  ../../../data/Arabic_training_6k_custom/vocab.txt
token :  custom
model :  ../../../checkpoints/model_150000.pt 



In [7]:
# inference process

def infer(ref_audio, ref_text, gen_text: str, spd: float = speed, nfe: int = nfe_step):

    generated_audio_segments = []
    reg1 = r"(?=\[\w+\])"
    chunks = re.split(reg1, gen_text)
    reg2 = r"\[(\w+)\]"
    for text in chunks:
        text = re.sub(reg2, "", text)
        gen_text_ = text.strip()

        audio_segment, final_sample_rate, spectrogram = infer_process(
            ref_audio,
            ref_text,
            gen_text_,
            ema_model,
            vocoder,
            mel_spec_type=vocoder_name,
            target_rms=target_rms,
            cross_fade_duration=cross_fade_duration,
            nfe_step=nfe,
            cfg_strength=cfg_strength,
            sway_sampling_coef=sway_sampling_coef,
            speed=spd,
            fix_duration=fix_duration,
            device=device,
        )
        generated_audio_segments.append(audio_segment)


    if generated_audio_segments:
        return np.concatenate(generated_audio_segments), final_sample_rate
    else:
        print("No audio segments generated.")
        return None

In [8]:
ref_audio = "examples/male_ae_00094.wav"
ref_text = extract_units(ref_audio, xeus_model, apply_kmeans, device)

In [9]:
src_wav = "examples/S04-C03-R04_004580-005165.wav"
src_text = extract_units(src_wav, xeus_model, apply_kmeans, device)

In [10]:
gen_wav , final_sample_rate= infer(ref_audio, ref_text, src_text, spd=1.0, nfe=12)

gen_text 0 󰆰󰘚󰍳󰋿󰀺󰏞󰑕󰀱󰍍󰘪󰍍󰘚󰍍󰔃󰍍󰖣󰀱󰍍󰏍󰍍󰔃󰝻󰙝󰘚󰙝󰋭󰑙󰙭󰒧󰚎󰚹󰚎󰋿󰞠󰅮󰌂󰈻󰝓󰁢󰑠󰅑󰌡󰌠󰆙󰝚󰘁󰂜󰇆󰌍󰜖󰓳󰆸󰁳󰅄󰋡󰘨󰚦󰆽󰋙󰜶󰁌󰚒󰈼󰃂󰁠󰀚󰈓󰄷󰌜󰜙󰑸󰆟󰛬󰇤󰎔󰇖󰐷󰍨󰋞󰒭󰁫󰓨󰖟󰈍󰍼󰔤󰒀󰔤󰒀󰀢󰒷󰄦󰅈󰂬󰜒󰔶󰂩󰁑󰈊󰖸󰙓󰌙󰋴󰏃󰈾󰊀󰂺󰙥󰃰󰛙󰜥󰕝󰅋󰕳󰁤󰞿󰊢󰄅󰝃󰛍󰍙󰜨󰂉󰜿󰌯󰈐󰓰󰈣󰅌󰁖󰖡󰍡󰀪󰈯󰕔󰛌󰘆󰈂󰀿󰊵󰑁󰁓󰄌󰏡󰘡󰐸󰞒󰙦󰝀󰁎󰂽󰙣󰜣󰂫󰉀󰞒󰗈󰘲󰝀󰘅󰍙󰆒󰎲󰜮󰋬󰙞󰖃󰋅󰘲󰞮󰂬󰜒󰙂󰁺󰃓󰀽󰝁󰏏󰞿󰇗󰋤󰄵󰘲󰗺󰘅󰆒󰎲󰘝󰚰󰋬󰕨󰔹󰙎󰚊󰊧󰍵󰃶󰈜󰎫󰈷󰞟󰘚󰆉󰋿󰄰󰋿󰙽󰝇󰂓󰖍󰂹󰎪󰅷󰀔󰋿󰄰󰄳󰋿󰊔󰋿󰄰󰖓󰂵󰋿󰂓󰔫󰖊󰔫󰀺󰋿󰙽󰖍󰋿󰊔󰖓󰎂


Generating audio in 1 batches...


100%|██████████| 1/1 [00:00<00:00,  1.16it/s]


In [11]:
# Target speaker
play(ref_audio)

In [12]:
# Source speech
play(src_wav )

In [13]:
display(play(gen_wav, rate=final_sample_rate))